In [ ]:
import sys
!{sys.executable} -m pip install nbformat>=4.2.0 ipywidgets scikit-learn

# EXP_009a2: The Lucier Resonance — Per-Layer Scan

## Scientific Objective (Plain Language)

**What are we doing?**
In EXP_009aFIX we looped the full tensor through ALL 12 layers at once (the whole brain). Here we isolate each layer individually: loop through just Layer 0, then just Layer 1, then just Layer 2, etc.

**What will we see?**
Each layer performs different operations on the data. Early layers typically handle syntax and position, middle layers handle factual recall, and late layers handle abstraction. By running the resonance loop on each layer in isolation, we create a map showing:
- Which layers are the strongest 'resonant filters' (fastest to converge)
- Which layers are the most 'transparent' (weakest, barely change the signal)
- Whether different layers attract different token archetypes

**Why does this matter?**
- Maps the functional specialization of the network layer by layer
- Identifies which layers contribute most to mode collapse / repetition traps
- Shows where the architecture's structural biases are concentrated

---

In [ ]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

In [ ]:
# ============================================================
# STEP 2: CONFIGURATION
# ============================================================

ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100, 250, 500]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

# Single probe prompt for the layer scan
PROBE_PROMPT = "The Eiffel Tower is located in the city of"

# We will test each layer as an individual 'room'
N_LAYERS = model.cfg.n_layers

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Probe prompt: '{PROBE_PROMPT}'")
print(f"Layers to scan: {N_LAYERS}")
print(f"MODE: PER-LAYER TOTAL RESONANCE")

In [ ]:
# ============================================================
# STEP 3: THE CORE ENGINE — Single-Layer Total Resonance
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies Final LayerNorm before unembedding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_single_layer_resonance(model, prompt, target_layer, max_iter, schedule):
    """
    Loop the ENTIRE residual stream tensor through a SINGLE layer.
    
    For target_layer L:
    - Read from: blocks.L.hook_resid_post (output of layer L)
    - Write to:  blocks.L.hook_resid_pre  (input of layer L)
    - The tensor loops through just this one layer repeatedly.
    
    Returns a list of snapshot dicts.
    """
    snapshots = []
    hook_read = f"blocks.{target_layer}.hook_resid_post"
    hook_write = f"blocks.{target_layer}.hook_resid_pre"
    
    # Iteration 0: Get the natural output of this layer
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_read
        )
    
    current_tensor = cache[hook_read][0].clone()  # [seq_len, d_model]
    seq_len = current_tensor.shape[0]
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        snapshots.append({
            "iteration": 0,
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "top_tokens": get_top_tokens(model, last_vec),
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_pos = current_tensor / pos_norms
            pos_sim = normalized_pos @ normalized_pos.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim.device)
            position_similarity = pos_sim[mask].mean().item()
            
            top_tokens = get_top_tokens(model, last_vec)
            
            snapshots.append({
                "iteration": i,
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "top_tokens": top_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Per-layer resonance engine loaded.")

In [ ]:
# ============================================================
# STEP 4: RUN THE EXPERIMENT — All 12 Layers
# ============================================================
from tqdm.notebook import tqdm

layer_results = {}

for layer in tqdm(range(N_LAYERS), desc="Scanning layers"):
    print(f"\n{'='*50}")
    print(f"LAYER {layer}: Resonance scan")
    print(f"{'='*50}")
    
    snapshots = run_single_layer_resonance(
        model, PROBE_PROMPT,
        target_layer=layer,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    layer_results[layer] = snapshots
    
    final = snapshots[-1]
    top_tok = final['top_tokens'][0][0].replace('\n', '\\n')
    print(f"  Final state: cos_mean={final['cosine_sim_mean']:.4f}, "
          f"pos_collapse={final['position_similarity']:.4f}, "
          f"top='{top_tok}'")

print(f"\n{'='*50}")
print("ALL LAYERS SCANNED.")

---
## 5. Visualization

### 5a. Layer Convergence Comparison
Overlay convergence curves for all 12 layers. Which layers resonate fastest?

In [ ]:
# ============================================================
# VIS 5a: CONVERGENCE CURVES — All Layers
# ============================================================

fig_conv = go.Figure()

for layer, snapshots in layer_results.items():
    iters = [s["iteration"] for s in snapshots]
    cos_mean = [s["cosine_sim_mean"] for s in snapshots]
    fig_conv.add_trace(go.Scatter(
        x=iters, y=cos_mean,
        mode='lines+markers',
        name=f"Layer {layer}",
        marker=dict(size=6),
    ))

fig_conv.update_layout(
    title="Per-Layer Resonance: Mean-Sequence Convergence",
    xaxis_title="Iteration",
    yaxis_title="Cosine Similarity (mean-seq to previous)",
    xaxis_type="log",
    template="plotly_dark",
    height=600,
)
fig_conv.add_hline(y=1.0, line_dash="dash", line_color="white", opacity=0.3)
fig_conv.show()

### 5b. Position Collapse Per Layer
Which layers cause the token positions to merge into identical vectors?

In [ ]:
# ============================================================
# VIS 5b: POSITION COLLAPSE — Per Layer
# ============================================================

fig_pos = go.Figure()

for layer, snapshots in layer_results.items():
    iters = [s["iteration"] for s in snapshots]
    pos_sim = [s["position_similarity"] for s in snapshots]
    fig_pos.add_trace(go.Scatter(
        x=iters, y=pos_sim,
        mode='lines+markers',
        name=f"Layer {layer}",
        marker=dict(size=6),
    ))

fig_pos.update_layout(
    title="Position Collapse Per Layer",
    xaxis_title="Iteration",
    yaxis_title="Mean Pairwise Cosine Similarity (All Positions)",
    xaxis_type="log",
    template="plotly_dark",
    height=600,
)
fig_pos.add_hline(y=1.0, line_dash="dash", line_color="white", opacity=0.3,
                  annotation_text="Total Collapse")
fig_pos.show()

### 5c. Layer Resonance Summary
Bar chart: final convergence score and resonant token per layer.

In [ ]:
# ============================================================
# VIS 5c: LAYER SUMMARY — Bar Chart + Token Labels
# ============================================================

layers = list(range(N_LAYERS))
final_cos = [layer_results[l][-1]["cosine_sim_mean"] for l in layers]
final_pos = [layer_results[l][-1]["position_similarity"] for l in layers]
final_tokens = []
for l in layers:
    tok = layer_results[l][-1]['top_tokens'][0][0]
    final_tokens.append(tok.replace('\n', '\\n').strip())

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=[f"L{l}" for l in layers],
    y=final_cos,
    name="Mean-Seq Convergence",
    text=[f"{c:.3f}" for c in final_cos],
    textposition='outside',
))
fig_bar.add_trace(go.Bar(
    x=[f"L{l}" for l in layers],
    y=final_pos,
    name="Position Collapse",
    text=[f"{p:.3f}" for p in final_pos],
    textposition='outside',
))

fig_bar.update_layout(
    title="Layer Resonance Summary at Iteration 500",
    xaxis_title="Layer",
    yaxis_title="Score",
    template="plotly_dark",
    height=500,
    barmode='group',
)
fig_bar.show()

# Token table
md = "### Resonant Token Per Layer\n\n"
md += "| Layer | Token | Mean-Seq Convergence | Position Collapse |\n"
md += "| :--- | :--- | :--- | :--- |\n"
for l in layers:
    md += f"| L{l} | `{final_tokens[l]}` | {final_cos[l]:.4f} | {final_pos[l]:.4f} |\n"
display(Markdown(md))

### 5d. 3D Topology — Per-Layer Trajectories
Project all 12 layer trajectories into the same 3D PCA space.

In [ ]:
# ============================================================
# VIS 5d: 3D PCA — Per-Layer Trajectories
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
iters_list = []
text_list = []

for layer, snapshots in layer_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(f"Layer {layer}")
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '\\n')
        text_list.append(f"L{layer} Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0], 'y': vecs_3d[:, 1], 'z': vecs_3d[:, 2],
    'Layer': labels_list, 'Iteration': iters_list, 'Info': text_list
})

fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Layer',
    hover_name='Info',
    markers=True,
    title=f"Per-Layer Topology: 3D Resonance Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=4), line=dict(width=3))
fig_topo.update_layout(
    template="plotly_dark", height=800,
    scene=dict(xaxis_title="PC 1", yaxis_title="PC 2", zaxis_title="PC 3")
)
fig_topo.show()

In [ ]:
# ============================================================
# STEP 6: SAVE ARTIFACTS
# ============================================================
import os

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

save_data = {}
for layer, snapshots in layer_results.items():
    save_data[f"L{layer}"] = {
        "iterations": [s["iteration"] for s in snapshots],
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(save_dir, "009a2_layer_scan_results.pt"))
print(f"[SAVED] {save_dir}/009a2_layer_scan_results.pt")